# Evaluating a Local LLM for Safe Warehouse Robot Planning

This notebook runs the dissertation experiments. It extends the professor's Qwen simulator with corrected relative movement, natural-language pick-and-deliver task planning, A* route ground truth, controlled difficulty, reproducible seeds, richer metrics and an exploratory hybrid LLM + A* comparison.

**Important:** first run a small pilot. Use the larger final sample only after checking the generated questions and outputs. Do not invent or manually change results.

## 1. Locate and install the project
In Google Colab, upload the project ZIP when the file chooser appears. In local Jupyter, first extract the ZIP and open this notebook from inside the `warehouse-llm-dissertation` folder; the cell will find the local project automatically.

In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Running in Google Colab. Upload the project ZIP.')
    uploaded = files.upload()
    zip_name = next(name for name in uploaded if name.endswith('.zip'))
    extraction_dir = Path('/content/warehouse_submission')
    extraction_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall(extraction_dir)
    PROJECT_DIR = extraction_dir / 'warehouse-llm-dissertation'
else:
    print('Running in local Jupyter. No google.colab package is required.')
    candidates = [Path.cwd(), Path.cwd() / 'warehouse-llm-dissertation', Path.cwd().parent]
    PROJECT_DIR = next((path for path in candidates if (path / 'pyproject.toml').exists()), None)
    if PROJECT_DIR is None:
        raise FileNotFoundError(
            'Extract the project ZIP and open this notebook from the warehouse-llm-dissertation folder.'
        )

PROJECT_DIR = PROJECT_DIR.resolve()
os.chdir(PROJECT_DIR)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_DIR)])
print('Project ready at:', PROJECT_DIR)

## 2. Verify the simulator before using an LLM
The oracle is only a pipeline check. It must score 100%; it is not an experimental result.

In [ ]:
!pytest -q
!python -m warehouse_llm.cli all --client oracle --cases 2 --output-dir results/oracle_check

## 3. Start Ollama and obtain Qwen
In Colab this cell installs Ollama. Locally, install the Ollama application first; this cell will detect it. Select a GPU runtime in Colab when available.

In [ ]:
import shutil, subprocess, time, urllib.request

if IN_COLAB:
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'zstd'], check=True)
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
elif shutil.which('ollama') is None:
    raise RuntimeError('Ollama is not installed locally. Install the Ollama app, then rerun this cell.')

try:
    urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
    print('Ollama server is already running.')
except Exception:
    ollama_process = subprocess.Popen(['ollama', 'serve'])
    time.sleep(5)

subprocess.run(['ollama', 'pull', 'qwen:7b'], check=True)
print('Qwen is ready.')

## 4. Pilot experiment
This produces 16 direction cases, 12 route cases and 12 pick-and-deliver mission cases. Inspect the CSV files before the final run.

In [ ]:
!python -m warehouse_llm.cli direction --client ollama --model qwen:7b --cases 1 --seed 202600 --output-dir results/pilot
!python -m warehouse_llm.cli route --client ollama --model qwen:7b --cases 1 --seed 202700 --output-dir results/pilot
!python -m warehouse_llm.cli mission --client ollama --model qwen:7b --cases 1 --seed 202800 --output-dir results/pilot

In [ ]:
import pandas as pd
from pathlib import Path
for csv_file in sorted(Path('results/pilot').glob('*.csv')):
    print('\n', csv_file)
    display(pd.read_csv(csv_file).head())

## 5. Final experiment
Thirty cases per direction condition gives 480 direction trials. Twenty cases per route condition gives 240 route trials, and twenty cases per task-planning condition gives 240 pick-and-deliver trials. Change these values only if your supervisor approves a different sample size. Keep the seeds fixed for reproducibility.

In [ ]:
DIRECTION_CASES_PER_CONDITION = 30
ROUTE_CASES_PER_CONDITION = 20
MISSION_CASES_PER_CONDITION = 20
!python -m warehouse_llm.cli direction --client ollama --model qwen:7b --cases $DIRECTION_CASES_PER_CONDITION --seed 202600 --output-dir results/final
!python -m warehouse_llm.cli route --client ollama --model qwen:7b --cases $ROUTE_CASES_PER_CONDITION --seed 202700 --output-dir results/final
!python -m warehouse_llm.cli mission --client ollama --model qwen:7b --cases $MISSION_CASES_PER_CONDITION --seed 202800 --output-dir results/final

## 6. Optional exploratory hybrid LLM + A* experiment
Qwen extracts only the requested action, pallet and dock. The verified A* planner generates the safe route. Keep this result separate from the original experiment.

In [ ]:
RUN_HYBRID_EXPERIMENT = False
HYBRID_CASES_PER_CONDITION = 20
if RUN_HYBRID_EXPERIMENT:
    !python -m warehouse_llm.cli hybrid --client ollama --model qwen:7b --cases $HYBRID_CASES_PER_CONDITION --seed 202800 --output-dir results/hybrid
else:
    print('Hybrid experiment not started. Change RUN_HYBRID_EXPERIMENT to True when ready.')

## 7. Analyse results
Calculate the actual accuracy, verify the sample structure, produce 95% confidence intervals, and create tables, graphs and a brief report.

In [ ]:
import subprocess, sys
analysis_arguments = ['analyse_results.py', '--input-dir', 'results/final', '--output-dir', 'results/analysis']
if Path('results/hybrid/hybrid_qwen-7b.csv').exists():
    analysis_arguments.extend(['--hybrid-dir', 'results/hybrid'])
subprocess.run([sys.executable, *analysis_arguments], check=True)

from IPython.display import Image, Markdown, display
analysis_dir = Path('results/analysis')
print('DATA-INTEGRITY CHECKS')
display(pd.read_csv(analysis_dir / 'experiment_integrity.csv'))
print('OVERALL ACCURACY TABLE')
display(pd.read_csv(analysis_dir / 'overall_accuracy_summary.csv'))
display(Markdown((analysis_dir / 'brief_accuracy_report.md').read_text()))
display(Image(filename=str(analysis_dir / 'accuracy_overview.png')))

## 8. Export the evidence package
Keep the raw CSV files, summaries, figure, executed notebook and software version together.

In [ ]:
import shutil
archive_path = shutil.make_archive('warehouse_llm_results', 'zip', 'results')
if IN_COLAB:
    files.download(archive_path)
else:
    print('Local results archive saved at:', Path(archive_path).resolve())